In [6]:
# Cài đặt thư viện underthesea cho tách từ tiếng Việt
!pip install underthesea
!pip install transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 8.1 MB/s eta 0:00:00


In [2]:
%cd "/content/drive/MyDrive/Colab Notebooks/ABSA_TMDT"
import os
print("Đang làm việc tại:", os.getcwd())
print("Danh sách các tệp tin:", os.listdir("."))

/content/drive/MyDrive/Colab Notebooks/ABSA_TMDT
Đang làm việc tại: /content/drive/MyDrive/Colab Notebooks/ABSA_TMDT
Danh sách các tệp tin: ['Untitled0.ipynb', '1-VLSP2018-SA-Hotel-train.csv', '2-VLSP2018-SA-Hotel-dev.csv', '3-VLSP2018-SA-Hotel-test.csv', '1-VLSP2018-SA-Restaurant-train.csv', '2-VLSP2018-SA-Restaurant-dev.csv', '3-VLSP2018-SA-Restaurant-test.csv']


In [3]:
import pandas as pd

# Đọc dữ liệu Hotel
hotel_train = pd.read_csv('1-VLSP2018-SA-Hotel-train.csv')
hotel_dev = pd.read_csv('2-VLSP2018-SA-Hotel-dev.csv')
hotel_test = pd.read_csv('3-VLSP2018-SA-Hotel-test.csv')

# Đọc dữ liệu Restaurant
restaurant_train = pd.read_csv('1-VLSP2018-SA-Restaurant-train.csv')
restaurant_dev = pd.read_csv('2-VLSP2018-SA-Restaurant-dev.csv')
restaurant_test = pd.read_csv('3-VLSP2018-SA-Restaurant-test.csv')

# Tạo cấu trúc lưu trữ gọn gàng
hotel_dataset = {
    'train': hotel_train,
    'dev': hotel_dev,
    'test': hotel_test
}

restaurant_dataset = {
    'train': restaurant_train,
    'dev': restaurant_dev,
    'test': restaurant_test
}

# In thông tin kiểm tra số lượng mẫu
print("=== Hotel Dataset ===")
for split, df in hotel_dataset.items():
    print(f"{split.capitalize()} set shape: {df.shape}")

print("\n=== Restaurant Dataset ===")
for split, df in restaurant_dataset.items():
    print(f"{split.capitalize()} set shape: {df.shape}")

=== Hotel Dataset ===
Train set shape: (3000, 35)
Dev set shape: (2000, 35)
Test set shape: (600, 35)

=== Restaurant Dataset ===
Train set shape: (2961, 13)
Dev set shape: (1290, 13)
Test set shape: (500, 13)


In [4]:
# Liệt kê tất cả các cột nhãn trong bộ dữ liệu Hotel
hotel_columns = list(hotel_train.columns)
print("Tổng số cột:", len(hotel_columns))
print("Cột đầu tiên:", hotel_columns[0])
print("Các cột nhãn gốc:")
for col in hotel_columns[1:]:
    print(f"- {col}")

Tổng số cột: 35
Cột đầu tiên: Review
Các cột nhãn gốc:
- FACILITIES#CLEANLINESS
- FACILITIES#COMFORT
- FACILITIES#DESIGN&FEATURES
- FACILITIES#GENERAL
- FACILITIES#MISCELLANEOUS
- FACILITIES#PRICES
- FACILITIES#QUALITY
- FOOD&DRINKS#MISCELLANEOUS
- FOOD&DRINKS#PRICES
- FOOD&DRINKS#QUALITY
- FOOD&DRINKS#STYLE&OPTIONS
- HOTEL#CLEANLINESS
- HOTEL#COMFORT
- HOTEL#DESIGN&FEATURES
- HOTEL#GENERAL
- HOTEL#MISCELLANEOUS
- HOTEL#PRICES
- HOTEL#QUALITY
- LOCATION#GENERAL
- ROOMS#CLEANLINESS
- ROOMS#COMFORT
- ROOMS#DESIGN&FEATURES
- ROOMS#GENERAL
- ROOMS#MISCELLANEOUS
- ROOMS#PRICES
- ROOMS#QUALITY
- ROOM_AMENITIES#CLEANLINESS
- ROOM_AMENITIES#COMFORT
- ROOM_AMENITIES#DESIGN&FEATURES
- ROOM_AMENITIES#GENERAL
- ROOM_AMENITIES#MISCELLANEOUS
- ROOM_AMENITIES#PRICES
- ROOM_AMENITIES#QUALITY
- SERVICE#GENERAL


In [5]:
# 1. Ánh xạ và Rút gọn dữ liệu (Data Mapping)

# Khai báo từ điển ánh xạ từ nhãn VLSP sang Khía cạnh CleanMate
aspect_mapping = {
    'Chất lượng': [
        'FACILITIES#CLEANLINESS', 'FACILITIES#QUALITY', 'FACILITIES#COMFORT', 'FACILITIES#DESIGN&FEATURES',
        'HOTEL#CLEANLINESS', 'HOTEL#QUALITY', 'HOTEL#COMFORT', 'HOTEL#DESIGN&FEATURES',
        'ROOMS#CLEANLINESS', 'ROOMS#QUALITY', 'ROOMS#COMFORT', 'ROOMS#DESIGN&FEATURES',
        'ROOM_AMENITIES#CLEANLINESS', 'ROOM_AMENITIES#QUALITY', 'ROOM_AMENITIES#COMFORT', 'ROOM_AMENITIES#DESIGN&FEATURES'
    ],
    'Nhân sự': ['SERVICE#GENERAL'],
    'Giá cả': ['FACILITIES#PRICES', 'HOTEL#PRICES', 'ROOMS#PRICES', 'ROOM_AMENITIES#PRICES'],
    'Thời gian': [] # Trong tập Hotel ít có nhãn thời gian rõ ràng, ta cứ để trống để cấu trúc đồng nhất
}

# Giả định quy ước nhãn của VLSP: 1 (Tích cực), 2 (Tiêu cực), 3 (Trung tính)
sentiments = {1: 'Tích cực', 2: 'Tiêu cực', 3: 'Trung tính'}

def map_to_cleanmate_format(df, aspect_mapping, sentiments):
    """
    Hàm chuyển đổi dataframe gốc sang định dạng Multi-label cho CleanMate.
    """
    # Tạo DataFrame mới chỉ giữ lại cột Review (bỏ qua LOCATION, FOOD&DRINKS...)
    new_df = pd.DataFrame()
    new_df['Review'] = df['Review']

    for target_aspect, source_cols in aspect_mapping.items():
        # Lọc các cột có tồn tại trong df
        valid_cols = [c for c in source_cols if c in df.columns]

        for sent_val, sent_name in sentiments.items():
            col_name = f"{target_aspect}_{sent_name}"
            if not valid_cols:
                # Nếu không có cột nguồn nào (VD: Thời gian), gán toàn bộ bằng 0
                new_df[col_name] = 0
            else:
                # Nếu review có ít nhất 1 cột nguồn chứa giá trị cảm xúc tương ứng -> 1, ngược lại -> 0
                mask = (df[valid_cols] == sent_val).any(axis=1)
                new_df[col_name] = mask.astype(int)

    return new_df

# Thực hiện ánh xạ cho các tập dữ liệu
cleanmate_train = map_to_cleanmate_format(hotel_dataset['train'], aspect_mapping, sentiments)
cleanmate_dev = map_to_cleanmate_format(hotel_dataset['dev'], aspect_mapping, sentiments)
cleanmate_test = map_to_cleanmate_format(hotel_dataset['test'], aspect_mapping, sentiments)

print("Kích thước tập Train mới:", cleanmate_train.shape)
print("Danh sách các cột (Khía cạnh_Cảm xúc):")
print(cleanmate_train.columns.tolist())

# Hiển thị dữ liệu mẫu
display(cleanmate_train.head(10))

Kích thước tập Train mới: (3000, 13)
Danh sách các cột (Khía cạnh_Cảm xúc):
['Review', 'Chất lượng_Tích cực', 'Chất lượng_Tiêu cực', 'Chất lượng_Trung tính', 'Nhân sự_Tích cực', 'Nhân sự_Tiêu cực', 'Nhân sự_Trung tính', 'Giá cả_Tích cực', 'Giá cả_Tiêu cực', 'Giá cả_Trung tính', 'Thời gian_Tích cực', 'Thời gian_Tiêu cực', 'Thời gian_Trung tính']


,Review,Chất lượng_Tích cực,Chất lượng_Tiêu cực,Chất lượng_Trung tính,Nhân sự_Tích cực,Nhân sự_Tiêu cực,Nhân sự_Trung tính,Giá cả_Tích cực,Giá cả_Tiêu cực,Giá cả_Trung tính,Thời gian_Tích cực,Thời gian_Tiêu cực,Thời gian_Trung tính
0,Rộng rãi KS mới nhưng rất vắng. Các dịch vụ ch...,1,0,0,0,0,0,0,0,0,0,0,0
1,"Địa điểm thuận tiện, trong vòng bán kính 1,5km...",0,0,0,0,0,0,0,0,0,0,0,0
2,"Phục vụ, view đẹp, vị trí",0,0,0,1,0,0,0,0,0,0,0,0
3,"thuận tiện , sạch sẽ , vui vẻ hài lòng",1,0,0,1,0,0,0,0,0,0,0,0
4,Vị trí đẹp; Có quán bar view đẹp; Nhân viên th...,0,0,0,1,0,0,0,0,0,0,0,0
5,- Co view huong Ho tay- sach se-nhan vien tan ...,1,0,0,1,0,0,0,0,0,0,0,0
6,"Phòng ốc sạch, giường thoải mái, nhân viên thâ...",1,0,0,1,0,0,0,0,0,0,0,0
7,"gần Hồ Tây, view nhìn ra hồ lãng mạn, đi bộ dọ...",1,0,0,0,0,0,0,1,0,0,0,0
8,Hình thức không thể quyết định nội dung.H&atil...,1,0,0,0,0,0,0,0,0,0,0,0
9,Tôi ở đây lần này là lần thứ 4. Khi nhận phòng...,0,1,0,1,0,0,0,0,0,0,0,0


In [7]:
import re
from underthesea import word_tokenize

def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    # 1. Chuyển về chữ thường
    text = text.lower()

    # 2. Loại bỏ các ký tự đặc biệt (chỉ giữ lại chữ cái, số và khoảng trắng)
    text = re.sub(r'[^\w\s]', ' ', text)

    # 3. Xóa khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text).strip()

    # 4. Tách từ tiếng Việt bằng underthesea
    text = word_tokenize(text, format="text")

    return text

# Áp dụng tiền xử lý cho cột Review
print("Đang xử lý tập Train...")
cleanmate_train['Review_clean'] = cleanmate_train['Review'].apply(preprocess_text)

print("Đang xử lý tập Dev...")
cleanmate_dev['Review_clean'] = cleanmate_dev['Review'].apply(preprocess_text)

print("Đang xử lý tập Test...")
cleanmate_test['Review_clean'] = cleanmate_test['Review'].apply(preprocess_text)

print("\nHoàn tất tiền xử lý! Dưới đây là kết quả so sánh:")
display(cleanmate_train[['Review', 'Review_clean']].head(5))

Đang xử lý tập Train...
Đang xử lý tập Dev...
Đang xử lý tập Test...

Hoàn tất tiền xử lý! Dưới đây là kết quả so sánh:


,Review,Review_clean
0,Rộng rãi KS mới nhưng rất vắng. Các dịch vụ ch...,rộng_rãi ks mới nhưng rất vắng các dịch_vụ chấ...
1,"Địa điểm thuận tiện, trong vòng bán kính 1,5km...",địa_điểm thuận_tiện trong vòng bán_kính 1 5 km...
2,"Phục vụ, view đẹp, vị trí",phục_vụ view đẹp vị_trí
3,"thuận tiện , sạch sẽ , vui vẻ hài lòng",thuận_tiện sạch_sẽ vui_vẻ hài_lòng
4,Vị trí đẹp; Có quán bar view đẹp; Nhân viên th...,vị_trí đẹp có quán bar view đẹp nhân_viên thân...


In [9]:
import torch
from transformers import AutoModel, AutoTokenizer
import torch.nn as nn

# Khởi tạo Tokenizer và pre-trained model PhoBERT
model_name = "vinai/phobert-base"
print(f"Đang tải tokenizer và model: {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
phobert = AutoModel.from_pretrained(model_name)

# Số lượng nhãn đầu ra (Khía cạnh_Cảm xúc)
# Trừ cột 'Review' và 'Review_clean'
num_labels = cleanmate_train.shape[1] - 2

class CleanMatePhoBERT(nn.Module):
    def __init__(self, phobert_model, num_labels):
        super(CleanMatePhoBERT, self).__init__()
        self.phobert = phobert_model
        # Dropout để giảm overfitting
        self.dropout = nn.Dropout(0.3)
        # Lớp Linear cuối cùng ánh xạ vector đặc trưng (768 chiều) sang số lượng nhãn
        self.classifier = nn.Linear(self.phobert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        # Lấy hidden states từ PhoBERT
        outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        # Lấy vector [CLS] đại diện cho toàn bộ câu
        pooled_output = outputs.pooler_output

        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

# Khởi tạo mô hình CleanMate
model = CleanMatePhoBERT(phobert, num_labels)
print(f"\nKhởi tạo thành công mô hình với {num_labels} nhãn đầu ra!")
print(model.classifier)

Đang tải tokenizer và model: vinai/phobert-base...


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Khởi tạo thành công mô hình với 12 nhãn đầu ra!
Linear(in_features=768, out_features=12, bias=True)


In [10]:
from torch.utils.data import Dataset, DataLoader

# Xác định các cột nhãn
label_cols = [col for col in cleanmate_train.columns if col not in ['Review', 'Review_clean']]
print("Các cột nhãn:", label_cols)

class CleanMateDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.texts = df['Review_clean'].tolist()
        self.labels = df[label_cols].values.tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = str(self.texts[index])
        labels = self.labels[index]

        # Tokenize văn bản
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.FloatTensor(labels)
        }

# Khởi tạo Datasets và DataLoaders
MAX_LEN = 256
BATCH_SIZE = 16

print("\nĐang tạo Dataset và DataLoader...")
train_dataset = CleanMateDataset(cleanmate_train, tokenizer, MAX_LEN)
dev_dataset = CleanMateDataset(cleanmate_dev, tokenizer, MAX_LEN)
test_dataset = CleanMateDataset(cleanmate_test, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Hoàn tất! Số lượng batch trong train_loader:", len(train_loader))

# Lấy thử 1 batch để kiểm tra
sample_batch = next(iter(train_loader))
print("Input IDs shape:", sample_batch['input_ids'].shape)
print("Labels shape:", sample_batch['labels'].shape)

Các cột nhãn: ['Chất lượng_Tích cực', 'Chất lượng_Tiêu cực', 'Chất lượng_Trung tính', 'Nhân sự_Tích cực', 'Nhân sự_Tiêu cực', 'Nhân sự_Trung tính', 'Giá cả_Tích cực', 'Giá cả_Tiêu cực', 'Giá cả_Trung tính', 'Thời gian_Tích cực', 'Thời gian_Tiêu cực', 'Thời gian_Trung tính']

Đang tạo Dataset và DataLoader...
Hoàn tất! Số lượng batch trong train_loader: 188
Input IDs shape: torch.Size([16, 256])
Labels shape: torch.Size([16, 12])


In [11]:
import torch.optim as optim
from tqdm import tqdm

# Thiết lập device (ưu tiên GPU nếu có)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sử dụng thiết bị: {device}")

# Chuyển mô hình sang thiết bị tính toán
model.to(device)

# Hàm mất mát cho bài toán Multi-label Classification
criterion = torch.nn.BCEWithLogitsLoss()

# Bộ tối ưu hóa AdamW
optimizer = optim.AdamW(model.parameters(), lr=2e-5)

Sử dụng thiết bị: cuda


In [12]:
EPOCHS = 3

print("Bắt đầu quá trình huấn luyện...")
for epoch in range(EPOCHS):
    # 1. Chế độ Huấn luyện (Training)
    model.train()
    total_train_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Xóa gradient cũ
        optimizer.zero_grad()

        # Forward pass (lan truyền xuôi)
        outputs = model(input_ids, attention_mask)

        # Tính toán loss
        loss = criterion(outputs, labels)
        total_train_loss += loss.item()

        # Backward pass (lan truyền ngược) & cập nhật trọng số
        loss.backward()
        optimizer.step()

    avg_train_loss = total_train_loss / len(train_loader)

    # 2. Chế độ Đánh giá (Validation)
    model.eval()
    total_val_loss = 0

    with torch.no_grad():
        for batch in tqdm(dev_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Dev]"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(dev_loader)

    # In kết quả sau mỗi Epoch
    print(f"\n--- Kết thúc Epoch {epoch+1} ---")
    print(f"Train Loss: {avg_train_loss:.4f} | Dev Loss: {avg_val_loss:.4f}\n")

Bắt đầu quá trình huấn luyện...




Epoch 1/3 [Train]:   0%|          | 0/188 [00:00<?, ?it/s]

Epoch 1/3 [Train]:   1%|          | 1/188 [00:04<13:50,  4.44s/it]

Epoch 1/3 [Train]:   1%|          | 2/188 [00:04<06:15,  2.02s/it]

Epoch 1/3 [Train]:   2%|▏         | 3/188 [00:05<04:12,  1.37s/it]

Epoch 1/3 [Train]:   2%|▏         | 4/188 [00:05<03:13,  1.05s/it]

Epoch 1/3 [Train]:   3%|▎         | 5/188 [00:06<02:41,  1.13it/s]

Epoch 1/3 [Train]:   3%|▎         | 6/188 [00:07<02:22,  1.27it/s]

Epoch 1/3 [Train]:   4%|▎         | 7/188 [00:07<02:12,  1.36it/s]

Epoch 1/3 [Train]:   4%|▍         | 8/188 [00:08<02:04,  1.44it/s]

Epoch 1/3 [Train]:   5%|▍         | 9/188 [00:08<01:57,  1.52it/s]

Epoch 1/3 [Train]:   5%|▌         | 10/188 [00:09<01:53,  1.57it/s]

Epoch 1/3 [Train]:   6%|▌         | 11/188 [00:10<01:50,  1.60it/s]

Epoch 1/3 [Train]:   6%|▋         | 12/188 [00:10<01:48,  1.62it/s]

Epoch 1/3 [Train]:   7%|▋         | 13/188 [00:11<01:46,  1.64it/s]

Epoch 1/3 [Train]:   7%|▋         | 14/188 [00:11<


--- Kết thúc Epoch 1 ---
Train Loss: 0.3323 | Dev Loss: 0.2478



Epoch 2/3 [Dev]: 100%|██████████| 125/125 [00:25<00:00,  4.99it/s]



--- Kết thúc Epoch 2 ---
Train Loss: 0.2234 | Dev Loss: 0.2049



Epoch 3/3 [Dev]: 100%|██████████| 125/125 [00:25<00:00,  4.97it/s]


--- Kết thúc Epoch 3 ---
Train Loss: 0.1763 | Dev Loss: 0.1660



Sau khi huấn luyện xong, chúng ta sẽ chuyển sang **Bước 5: Đánh giá (Evaluation)** trên tập Test để đo lường độ chính xác (F1-score, Precision, Recall) cho từng nhãn. Hãy chạy huấn luyện và cho mình biết khi nào bạn sẵn sàng đi tiếp nhé!

In [13]:
import numpy as np
from sklearn.metrics import classification_report

print("Bắt đầu đánh giá trên tập Test...")
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating [Test]"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask)

        # Sử dụng hàm sigmoid để đưa output về dải [0, 1] cho bài toán multi-label
        probs = torch.sigmoid(outputs)

        # Phân loại dựa trên ngưỡng 0.5
        preds = (probs > 0.5).int()

        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

# Gộp kết quả từ tất cả các batch
all_preds = np.vstack(all_preds)
all_labels = np.vstack(all_labels)

# Báo cáo chi tiết
print("\n--- Báo cáo Phân loại (Classification Report) ---")
print(classification_report(all_labels, all_preds, target_names=label_cols, zero_division=0))

Bắt đầu đánh giá trên tập Test...


Evaluating [Test]: 100%|██████████| 38/38 [00:07<00:00,  4.87it/s]


--- Báo cáo Phân loại (Classification Report) ---
                       precision    recall  f1-score   support

  Chất lượng_Tích cực       0.76      0.94      0.84       408
  Chất lượng_Tiêu cực       0.77      0.86      0.81       265
Chất lượng_Trung tính       0.00      0.00      0.00        46
     Nhân sự_Tích cực       0.93      0.96      0.95       349
     Nhân sự_Tiêu cực       0.93      0.36      0.52        39
   Nhân sự_Trung tính       0.00      0.00      0.00        28
      Giá cả_Tích cực       0.94      0.71      0.81        91
      Giá cả_Tiêu cực       0.00      0.00      0.00        18
    Giá cả_Trung tính       0.00      0.00      0.00         4
   Thời gian_Tích cực       0.00      0.00      0.00         0
   Thời gian_Tiêu cực       0.00      0.00      0.00         0
 Thời gian_Trung tính       0.00      0.00      0.00         0

            micro avg       0.82      0.82      0.82      1248
            macro avg       0.36      0.32      0.33      1248
  

In [15]:
def predict_review(text, model, tokenizer, device, threshold=0.5):
    # 1. Tiền xử lý
    clean_text = preprocess_text(text)

    # 2. Tokenize
    encoding = tokenizer(
        clean_text,
        add_special_tokens=True,
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt',
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # 3. Dự đoán
    model.eval()
    with torch.no_grad():
        outputs = model(input_ids, attention_mask)
        probs = torch.sigmoid(outputs).squeeze(0) # Loại bỏ chiều batch

    # 4. Lấy các nhãn vượt ngưỡng (threshold)
    predictions = []
    for idx, prob in enumerate(probs):
        if prob > threshold:
            predictions.append((label_cols[idx], prob.item()))

    return predictions

# --- Thử nghiệm trên một số câu mới ---
sample_reviews = [
    "Khách sạn sạch sẽ, giường ngủ êm ái nhưng nhân viên lễ tân hơi lạnh lùng.",
    "Giá phòng quá cao so với chất lượng, wifi yếu.",
    "Phòng đẹp, view nhìn ra biển rất tuyệt, nhân viên hỗ trợ nhiệt tình, tôi sẽ quay lại.",
    "Phòng vệ sinh bẩn, mùi hôi khó chịu. Mình phải gọi dọn phòng ngay lập tức."
]

print("=== KẾT QUẢ DỰ ĐOÁN ===")
for review in sample_reviews:
    print(f"\nReview: \"{review}\"")
    preds = predict_review(review, model, tokenizer, device)
    if preds:
        for label, prob in preds:
            print(f"  -> {label}")
    else:
        print("  -> [Không phát hiện khía cạnh/cảm xúc nào]")

=== KẾT QUẢ DỰ ĐOÁN ===

Review: "Khách sạn sạch sẽ, giường ngủ êm ái nhưng nhân viên lễ tân hơi lạnh lùng."
  -> Chất lượng_Tiêu cực

Review: "Giá phòng quá cao so với chất lượng, wifi yếu."
  -> Chất lượng_Tiêu cực

Review: "Phòng đẹp, view nhìn ra biển rất tuyệt, nhân viên hỗ trợ nhiệt tình, tôi sẽ quay lại."
  -> Chất lượng_Tích cực
  -> Nhân sự_Tích cực

Review: "Phòng vệ sinh bẩn, mùi hôi khó chịu. Mình phải gọi dọn phòng ngay lập tức."
  -> Chất lượng_Tiêu cực


In [17]:
house_cleaning_reviews = [
    "Nhân viên dọn dẹp rất sạch sẽ, lau nhà kỹ càng và sắp xếp đồ đạc gọn gàng.",
    "Giá dịch vụ vệ sinh nhà cửa khá đắt so với mặt bằng chung.",
    "Người giúp việc đến trễ, làm việc lại còn ẩu, quét nhà không sạch.",
    "Dụng cụ lau dọn vệ sinh rất hiện đại, tôi rất hài lòng thái độ của nhân viên.",
    "Nhà vệ sinh cọ rửa chưa sạch, vẫn còn mùi hôi."
]

print("=== KẾT QUẢ DỰ ĐOÁN (LĨNH VỰC VỆ SINH NHÀ CỬA) ===")
for review in house_cleaning_reviews:
    print(f"\nReview: \"{review}\"")
    preds = predict_review(review, model, tokenizer, device)
    if preds:
        for label, prob in preds:
            print(f"  -> {label}")
    else:
        print("  -> [Không phát hiện khía cạnh/cảm xúc nào]")

=== KẾT QUẢ DỰ ĐOÁN (LĨNH VỰC VỆ SINH NHÀ CỬA) ===

Review: "Nhân viên dọn dẹp rất sạch sẽ, lau nhà kỹ càng và sắp xếp đồ đạc gọn gàng."
  -> Chất lượng_Tích cực
  -> Nhân sự_Tích cực

Review: "Giá dịch vụ vệ sinh nhà cửa khá đắt so với mặt bằng chung."
  -> [Không phát hiện khía cạnh/cảm xúc nào]

Review: "Người giúp việc đến trễ, làm việc lại còn ẩu, quét nhà không sạch."
  -> Chất lượng_Tiêu cực

Review: "Dụng cụ lau dọn vệ sinh rất hiện đại, tôi rất hài lòng thái độ của nhân viên."
  -> Chất lượng_Tích cực
  -> Nhân sự_Tích cực

Review: "Nhà vệ sinh cọ rửa chưa sạch, vẫn còn mùi hôi."
  -> Chất lượng_Tiêu cực


In [18]:
import os

def save_model(model, save_path="cleanmate_model.pth"):
    """Lưu trọng số của mô hình"""
    torch.save(model.state_dict(), save_path)
    print(f"[INFO] Đã lưu trọng số mô hình tại: {save_path}")

def load_model(model, load_path="cleanmate_model.pth", device='cpu'):
    """Tải trọng số vào kiến trúc mô hình"""
    if not os.path.exists(load_path):
        print(f"[ERROR] Không tìm thấy file {load_path}")
        return model

    model.load_state_dict(torch.load(load_path, map_location=device))
    model.to(device)
    model.eval() # Chuyển sang chế độ đánh giá
    print(f"[INFO] Đã tải thành công trọng số từ: {load_path}")
    return model

# ==========================================
# DEMO: LƯU, TẢI VÀ DỰ ĐOÁN
# ==========================================

# 1. Lưu mô hình bạn vừa huấn luyện xong
save_file = "cleanmate_phobert_v1.pth"
save_model(model, save_file)

# 2. Khởi tạo một kiến trúc mô hình mới (chưa có trọng số đã huấn luyện)
print("\n[INFO] Khởi tạo một mô hình mới để test...")
# Lưu ý: 'phobert' và 'num_labels' đã được định nghĩa ở các ô code trước
loaded_model = CleanMatePhoBERT(phobert, num_labels)

# 3. Tải trọng số từ file vừa lưu vào mô hình mới
loaded_model = load_model(loaded_model, save_file, device)

# 4. Test thử mô hình vừa được load lên
test_sentence = "Dịch vụ quá tệ, dọn dẹp không sạch mà lấy giá trên trời!"
print(f"\n=== TEST MÔ HÌNH LOAD TỪ FILE ===")
print(f"Review: \"{test_sentence}\"")

# Dùng hàm predict_review đã viết ở trên (truyền vào loaded_model thay vì model)
preds = predict_review(test_sentence, loaded_model, tokenizer, device)
if preds:
    for label, prob in preds:
        print(f"  -> {label} (Độ tự tin: {prob*100:.1f}%)")
else:
    print("  -> [Không phát hiện khía cạnh/cảm xúc nào]")

[INFO] Đã lưu trọng số mô hình tại: cleanmate_phobert_v1.pth

[INFO] Khởi tạo một mô hình mới để test...
[INFO] Đã tải thành công trọng số từ: cleanmate_phobert_v1.pth

=== TEST MÔ HÌNH LOAD TỪ FILE ===
Review: "Dịch vụ quá tệ, dọn dẹp không sạch mà lấy giá trên trời!"
  -> Chất lượng_Tiêu cực (Độ tự tin: 64.4%)
